In [1]:
import geogram as geo
import numpy as np
from tqdm import trange, tqdm

import matplotlib.pyplot as plt
import matplotlib.tri as tri
import matplotlib.collections

import scipy

# Note that we freeze the seed to ensure everyone will have the same results.
np.random.seed(0)

In [2]:
def triangle_area(vertices, v1, v2, v3):
    """
    @brief Computes the area of a mesh triangle
    @param[in] vertices the coordinates of the mesh vertices
    @param[in] triangles an array with the three vertices indices of the triangle
    @details Works also when T is an array of triangles (then it returns
        the array of triangle areas). This is why the ellipsis (...)
        is used (here it means indexing/slicing through the last dimension)
        Use a formula which is independent from the dimension 
    """
    a = np.sqrt(np.sum((vertices[v2] - vertices[v1])**2, axis=1))
    b = np.sqrt(np.sum((vertices[v3] - vertices[v1])**2, axis=1))
    c = np.sqrt(np.sum((vertices[v3] - vertices[v2])**2, axis=1))

    s = .5 * (a + b + c)
    return np.sqrt(s * (s-a) * (s-b) * (s-c))


def tet_volume(vertices, tetrahedron):
    v1 = tetrahedron[..., 0]
    v2 = tetrahedron[..., 1]
    v3 = tetrahedron[..., 2]
    v4 = tetrahedron[..., 3]

    a = vertices[v1] - vertices[v4]
    b = vertices[v2] - vertices[v4]
    c = vertices[v3] - vertices[v4]

    return -(1. / 6.) * (a * np.cross(b, c)).sum(axis=1)


def distance(vertices, v1, v2):
    """
    @brief Computes the length of a mesh edge
    @param[in] vertices the coordinates of the mesh vertices
    @param[in] v1, v2 the mesh extremities indices.
    @details v1 and v2 can be also arrays (then returns the array of distances).
    """
    axis = v1.ndim if hasattr(v1, 'ndim') else 0
    return np.linalg.norm(vertices[v2] - vertices[v1], axis=axis)


def hessian(voronoi):
    """
    @brief Assembles the Hessian of the Kantorovich dual
    @param[in] voronoi geo.Voronoi the voronoi diagram
    @return I,J,VAL row,column,value arrays, with the extra-diagonal coeffs
    @details One needs to compute the diagonal (= -sum of extra-diagonal coeffs)
    """

    NO_INDEX = -1  # Special value for invalid indices (edge on border)

    # i is the seed associated with the triangle
    # j is the seed on the other side of the triangle's edge (NO_INDEX on border)
    i = voronoi.tseed
    tadj = voronoi.tadj

    j = tadj.T.flatten().astype(np.int32)
    j = np.where(j != NO_INDEX, i[j], NO_INDEX).astype(np.int32)

    i  = np.concatenate((i, i, i, i))
    v1 = np.concatenate((voronoi.t[:, 1], voronoi.t[:, 2], voronoi.t[:, 3], voronoi.t[:, 0]))
    v2 = np.concatenate((voronoi.t[:, 2], voronoi.t[:, 3], voronoi.t[:, 0], voronoi.t[:, 1]))
    v3 = np.concatenate((voronoi.t[:, 3], voronoi.t[:, 0], voronoi.t[:, 1], voronoi.t[:, 2]))

    qidx = np.column_stack((i, j, v1, v2, v3))
    qidx = qidx[np.logical_and(i != j, j != NO_INDEX)]

    i = qidx[:, 0]  # re-extract i, j, v1, v2
    j = qidx[:, 1]
    v1 = qidx[:, 2]
    v2 = qidx[:, 3]
    v3 = qidx[:, 4]

    val = -triangle_area(voronoi.q, v1, v2, v3) / (2. * distance(voronoi.seeds, i, j))
    
    return i, j, val


def measures(voronoi):
    """
    @brief Computes the measures of the Laguerre cells
    @return the vector of Laguerre cells measures
    @details Uses the current Laguerre diagram (in self.Laguerre)
    """
    # See comments about XY,T,trgl_seed,nt in compute_Laguerre_diagram()
    measures = np.zeros(len(voronoi.seeds))
    np.add.at(measures, voronoi.tseed, tet_volume(voronoi.q, voronoi.t))
    return measures

In [3]:
class Transport:
    def __init__(self, domain=None, use_direct_solver: bool = True, verbose: bool = False):
        """
        @brief Transport constructor
        @param[in] seeds seeds coordinates
        @param[in] domain Voronoi diagram domain
        @param[in] use_direct_solver: direct (if set) or iterative solver otherwise
        @param[in] verbose log Newton iterations if set
        """

        self.direct = use_direct_solver
        self.verbose = verbose

        self.domain = domain
        if self.domain is None:
            (vertices, triangles) = geo.shape.cube()
            self.domain           = geo.mesh.tetrahedralize(vertices, triangles)

        # Parameters for linear solver
        self.regularization = 0.0
        if self.direct:  # if using direct solver, one needs regulariz.
            self.regularization = 1e-6  # because matrix is singular ([1,1...1] in ker)


    def solve(self, seeds, relative_error=0.01):
        dimension = seeds.shape[1]
        assert dimension == 3, 'seeds must be a (N, 3) array.'

        # Compute Laguerre diagram
        psi     = np.zeros(len(seeds), np.float64)
        voronoi = geo.Voronoi(seeds, psi, domain_vertices=self.domain[0], domain_simplices=self.domain[1])

        # Measure of whole domain, desired areas and minimum legal area (KMT #1)
        areas = measures(voronoi)
        assert np.min(areas) > 0., 'Voronoi diagram has an empty cell.'

        omega_measure = np.sum(areas)  # Measure of the whole domain
        nu_i = omega_measure / len(seeds)  # Desired area for each cell
        area_threshold = 0.5 * min(np.min(areas), nu_i)  # KMT criterion  #1

        threshold = nu_i * relative_error  # 1% of desired cell area
        current_error = np.inf

        i = 0
        while current_error > threshold and i < 100:
            if self.verbose:
                print(f'{i} New Newton step - current_error {current_error}' + '-'*10)

            H = self.hessian(seeds, voronoi, nu_i)  # Hessian of Kantorovich dual (sparse matrix)

            # rhs (minus gradient of Kantorovich dual) = desired areas - actual areas
            b = nu_i - measures(voronoi)
            if self.regularization != 0.0:
                b -= self.regularization * nu_i * psi

            g_norm = np.linalg.norm(b)  # norm of gradient at current step (KMT #2)
            p = self.solve_linear_system(H, b)  # solve for p in H*p=b
            alpha = 1.0  # Steplength
            psi += p  # Start with Newton step

            # Divide steplength by 2 until both KMT criteria are satisfied
            for ii in range(100):
                voronoi = geo.Voronoi(seeds, psi, domain_vertices=self.domain[0], domain_simplices=self.domain[1])

                # g (grad of Kantorovich dual) at substep = actual areas - desired areas
                g = measures(voronoi)
                smallest_area = np.min(g)  # for KMT criterion 1
                g -= nu_i

                # Check KMT criteria #1 (cell area) and #2 (gradient norm)
                g_norm_k = np.linalg.norm(g)
                kmt_1 = (smallest_area > area_threshold)  # criterion 1: cell area
                kmt_2 = (g_norm_k <= (1.0 - 0.5 * alpha) * g_norm)  # criterion 2: gradient norm
                if kmt_1 and kmt_2:
                    break

                alpha = alpha / 2.0
                psi -= alpha * p

            if ii == 100:
                print('Error: Did not converged!')
            
            if self.verbose:
                print(f'Alpha found {alpha} in {ii} steps')
            
            current_error = np.linalg.norm(b, ord=np.inf)
            i += 1

        if i == 100:
            print('Error: Did not converged!')
        
        return voronoi

    def solve_linear_system(self, H, b):
        """
        @brief Solves a linear system
        @details Works in direct or iterative mode, with scipy and with OpenNL
        @param[in] H the matrix of the linear system
        @param[in] b the right hand side
        @return p such that H p = b
        """
        if self.direct:
            p = scipy.sparse.linalg.spsolve(H, b)
        else:
            linalg = scipy.sparse.linalg
            dim = (len(b), len(b))

            # A: operator:       y <- (H + diag)*x
            # M: preconditioner: y <- diag@{-1}*x
            self.iter = 0
            p, info = linalg.cg(
                A=linalg.LinearOperator(dim, matvec=lambda x: H @ x + H.diag * x),
                b=b,
                M=linalg.LinearOperator(dim, matvec=lambda x: x / H.diag),
                callback=print if self.verbose else None,
                atol=0.0,  # normally the default, but larger on older scipy ver.
                rtol=1e-3  # or tol=1e-3 instead of rtol, depends on scipy ver.
            )
            if info != 0:
                print(f'CG did not converge, info={info}', info)
        return p


    def hessian(self, seeds, voronoi, nu_i):
        """
        @brief Computes the matrix of the system to be solved at each Newton step
        @details Uses the current Laguerre diagram (in self.Laguerre). Works in
         scipy and in OpenNL mode. In the (scipy,iterative) combination, the
         diagonal of the matrix is stored separately in a dynamically created
         'diag' field of the returned scipy sparse matrix.
        @return the Hessian matrix of the Kantorovich dual
        """

        i, j, val = hessian(voronoi)

        n = len(seeds)
        diag = np.zeros(n, np.float64)  # Diagonal (initialized to zero)
        np.add.at(diag, i, -val)  # =minus sum extra-diagonal coefficients
        if self.regularization != 0.0:
            diag += self.regularization * nu_i

        H = scipy.sparse.csr_array((val, (i, j)), shape=(n, n))
        if self.direct:  # if using direct solver, inject diag coeffs into mtx
            s = np.arange(n, dtype=np.int32)
            H += scipy.sparse.csr_array((diag, (s, s)), shape=(n, n))
        else:
            H.diag = diag  # store diagonal separately if using iterative solver

        return H

In [4]:
SEED_NB = 1000
seeds   = np.random.uniform(size=(SEED_NB, 3))

# Load and compute a volume domain
(vertices, triangles) = geo.shape.cube()
(vertices, tets)      = geo.mesh.tetrahedralize(vertices, triangles)

optimizer = Transport((vertices, tets), verbose=False)


import datetime
starttime = datetime.datetime.now()

optimized = optimizer.solve(seeds * .1, relative_error=1e-6)

print('-'*10)
print(f'Total elapsed time for OT: {datetime.datetime.now() - starttime}')

----------
Total elapsed time for OT: 0:00:09.493121


In [5]:
import polyscope as ps

ps.init()

# Display both meshes
voronoi = geo.Voronoi(seeds * .1, domain_vertices=vertices, domain_simplices=tets)

ps_vol = ps.register_volume_mesh("Start", voronoi.q, voronoi.t)
ps_vol.add_scalar_quantity("Cell indices", voronoi.tseed, enabled=True, defined_on='cells')
ps_vol.set_cull_whole_elements(True)

ps_vol = ps.register_volume_mesh("Optimized", optimized.q, optimized.t)
ps_vol.add_scalar_quantity("Cell indices", optimized.tseed, enabled=True, defined_on='cells')
ps_vol.set_position((1.1, 0, 0))

ps.set_ground_plane_mode('none')
ps.set_view_from_json("{\"farClip\":20.0,\"fov\":26.2520866394043,\"frontDir\":\"Z Forward\",\"navigateStyle\":\"Turntable\",\"nearClip\":0.00499999988824129,\"projectionMode\":\"Orthographic\",\"upDir\":\"Y Up\",\"viewCenter\":[1.05058491230011,0.445226579904556,0.516272068023682],\"viewMat\":[0.946401834487915,0.0,-0.322991669178009,-0.82752388715744,-0.0529155284166336,0.98648875951767,-0.155048429965973,-0.30357164144516,0.318627655506134,0.163829386234283,0.933614730834961,-3.79534482955933,0.0,0.0,0.0,1.0],\"viewRelativeMode\":\"Center Relative\",\"windowHeight\":1102,\"windowWidth\":1800}")
ps.show()

[polyscope] Backend: openGL3_glfw -- Loaded openGL version: 4.1 Metal - 90.5
